# Bengali Handwritten OCR Fine-Tuning with Qwen2-VL (QLoRA) on Google Colab
Train on the BN-HTRd dataset using 4-bit NF4 QLoRA on a free Google Colab T4 GPU (~15 GB VRAM).

> **Critical Setup**: Ensure your runtime is set to **T4 GPU** before running (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU found! Set Runtime -> Change runtime type -> T4 GPU.'
print(f'Active GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. Mount Google Drive for persistent checkpoints
# All checkpoints land directly in Drive so progress survives session disconnects
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/bangla_ocr_checkpoints

In [ ]:
# 3. Install required packages (Do NOT reinstall torch)
!pip install -q "transformers>=4.45.0" "accelerate>=0.30.0" "peft>=0.11.0" "bitsandbytes>=0.43.0" "qwen-vl-utils>=0.0.8" datasets pandas openpyxl pillow jiwer gdown kaggle

In [ ]:
# 4. Clone repository & Setup workspace
import os
%cd /content
if not os.path.exists('/content/bangla-ocr-qwen2vl'):
    !git clone https://github.com/RahulIslam46/bangla-ocr-qwen2vl.git /content/bangla-ocr-qwen2vl
%cd /content/bangla-ocr-qwen2vl
!git pull
!mkdir -p /content/data

In [ ]:
# 5. Download and Extract BN-HTRd Benchmark Dataset (~1.38 GB)
import os, glob, zipfile

data_dir = '/content/data/Dataset'
zip_path = '/content/BN-HTRd.zip'

if not os.path.exists(data_dir) or len(glob.glob(f'{data_dir}/*')) == 0:
    if not os.path.exists(zip_path):
        print('Downloading BN-HTRd dataset (~1.38 GB) from Google Drive...')
        !gdown --id 16eenWCoi4MahIJmIJ4iYPAGK9aL8X1Xy -O /content/BN-HTRd.zip
    
    print('Extracting outer archive...')
    !unzip -q -o /content/BN-HTRd.zip -d /content/data/
    nested = glob.glob('/content/data/**/Dataset.zip', recursive=True)
    if nested:
        print('Extracting inner Dataset.zip...')
        !unzip -q -o "{nested[0]}" -d /content/data/

print(f'Target dataset directory: {data_dir}')
if os.path.exists(data_dir):
    folders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
    print(f'Total document folders: {len(folders)}')

In [ ]:
# 6. Fetch Checkpoint-2001 (Resumes from Step 2,001 / 4,593)
import os, shutil

cp_dir = '/content/drive/MyDrive/bangla_ocr_checkpoints/checkpoint-2001'
if not os.path.exists(cp_dir) or not os.path.exists(os.path.join(cp_dir, 'adapter_model.safetensors')):
    print('Downloading checkpoint-2001 from Kaggle dataset (~108 MB)...')
    os.environ['KAGGLE_USERNAME'] = 'mcjdcu'
    os.environ['KAGGLE_KEY'] = '54231abd0f5c0b50ce32bd5e2d111e61'
    !kaggle datasets download -d mcjdcu/bangla-ocr-checkpoint-2001 -p /content/temp_cp --unzip
    os.makedirs(cp_dir, exist_ok=True)
    for f in os.listdir('/content/temp_cp'):
        shutil.copy2(os.path.join('/content/temp_cp', f), os.path.join(cp_dir, f))
    print('Checkpoint-2001 successfully saved to Google Drive!')
else:
    print('Checkpoint-2001 already present in Google Drive.')
print('Files in checkpoint:', os.listdir(cp_dir))

In [ ]:
# 7. Launch Stage 3 Training
!python train.py \
    --data_dir /content/data/Dataset \
    --mode line \
    --output_dir /content/drive/MyDrive/bangla_ocr_checkpoints \
    --epochs 3 \
    --batch_size 1 \
    --grad_accum 8 \
    --lr 2e-4 \
    --save_steps 50 \
    --logging_steps 5 \
    --max_val_samples 150 \
    --max_time_hours 10.5 \
    --resume_from_checkpoint /content/drive/MyDrive/bangla_ocr_checkpoints/checkpoint-2001

In [ ]:
# 8. Resume from latest Drive checkpoint (if session ever drops)
import glob
all_cps = sorted(glob.glob('/content/drive/MyDrive/bangla_ocr_checkpoints/checkpoint-*'),
                key=lambda x: int(x.split('-')[-1]) if x.split('-')[-1].isdigit() else 0)
if all_cps:
    latest = all_cps[-1]
    print(f'Resuming from latest checkpoint: {latest}')
    !python train.py \
        --data_dir /content/data/Dataset \
        --mode line \
        --output_dir /content/drive/MyDrive/bangla_ocr_checkpoints \
        --epochs 3 \
        --batch_size 1 \
        --grad_accum 8 \
        --lr 2e-4 \
        --save_steps 50 \
        --logging_steps 5 \
        --max_val_samples 150 \
        --max_time_hours 10.5 \
        --resume_from_checkpoint "{latest}"

In [ ]:
# 9. Verify Final Model & Evaluation
!python evaluate.py \
    --data_dir /content/data/Dataset \
    --adapter_path /content/drive/MyDrive/bangla_ocr_checkpoints/final_model \
    --num_samples 10